# Weakly-Supervised Bone Segmentation — FracAtlas
### Pipeline: DenseNet121 → LayerCAM → SAM ViT-B → U-Net

**Stages:**
1. Setup & Dataset Download
2. Stage 1 — Anatomy Classifier Training
3. Stage 2 — Pseudo Mask Generation (LayerCAM + SAM)
4. Stage 3 — U-Net Segmentation Training
5. Inference & Visualization

## 1. Setup

In [ ]:
# Mount Google Drive — outputs will be saved here across sessions
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo (skip if already cloned)
import os
if not os.path.exists('/content/Thesis'):
    !git clone https://github.com/itsthang333/Thesis.git /content/Thesis
else:
    print('Repo already exists, pulling latest...')
    !git -C /content/Thesis pull

%cd /content/Thesis/project
!pwd

In [ ]:
# Install dependencies
!pip install -q torch torchvision numpy pillow tqdm opencv-python kagglehub
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
print('All dependencies installed.')

In [ ]:
# Download FracAtlas dataset from Kaggle
import kagglehub

DATASET_PATH = kagglehub.dataset_download('mahmudulhasantasin/fracatlas-original-dataset')
print(f'Dataset path: {DATASET_PATH}')

# Verify structure
import os
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    if level < 2:
        print('  ' * level + os.path.basename(root) + '/')
    if level == 1:
        print('  ' * (level + 1) + f'({len(files)} files)')

In [ ]:
# Download SAM ViT-B checkpoint (~375 MB)
# Saved to Drive so it persists across Colab sessions
import os, urllib.request

SAM_CHECKPOINT_DIR = '/content/drive/MyDrive/ThesisOutputs/checkpoints'
SAM_CHECKPOINT_PATH = os.path.join(SAM_CHECKPOINT_DIR, 'sam_vit_b_01ec64.pth')
SAM_DOWNLOAD_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'

os.makedirs(SAM_CHECKPOINT_DIR, exist_ok=True)

if os.path.exists(SAM_CHECKPOINT_PATH):
    print(f'SAM checkpoint already exists: {SAM_CHECKPOINT_PATH}')
else:
    print('Downloading SAM ViT-B checkpoint (~375 MB)...')
    urllib.request.urlretrieve(SAM_DOWNLOAD_URL, SAM_CHECKPOINT_PATH)
    print(f'Saved to {SAM_CHECKPOINT_PATH}')

## 2. Stage 1 — Anatomy Classifier Training

Train DenseNet121 with BCEWithLogitsLoss on 4 anatomy labels: `hand, leg, hip, shoulder`.

In [ ]:
CLASSIFIER_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/classifier'

!python train_classifier.py \
  --data-root "{DATASET_PATH}" \
  --target-columns hand,leg,hip,shoulder \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 2 \
  --epochs 25 \
  --output-dir "{CLASSIFIER_OUTPUT}"

## 3. Stage 2 — Pseudo Mask Generation (LayerCAM + SAM)

Pipeline:
- **LayerCAM** on denseblock2/3/4 → confidence-filtered weighted fusion
- **Adaptive threshold** (percentile 85) → peak prompts
- **SAM ViT-B** generates candidate masks from prompts
- **CAM-guided selection** (mean CAM score ≥ 0.4) + logical-OR fusion
- **Morphological refinement** (closing → opening → fill_holes → remove_small)

In [ ]:
PSEUDO_MASK_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/pseudo_masks'
CLASSIFIER_CHECKPOINT = f'{CLASSIFIER_OUTPUT}/best_classifier.pt'

!python generate_pseudo_masks.py \
  --data-root "{DATASET_PATH}" \
  --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
  --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
  --target-columns hand,leg,hip,shoulder \
  --image-size 384 \
  --batch-size 1 \
  --num-workers 2 \
  --confidence-threshold 0.5 \
  --cam-percentile 85.0 \
  --max-points 5 \
  --min-component-area 100 \
  --mask-score-threshold 0.4 \
  --closing-kernel 5 \
  --opening-kernel 3 \
  --min-size 200 \
  --output-dir "{PSEUDO_MASK_OUTPUT}"

In [ ]:
# Sanity check: count generated masks and show a sample
import glob, os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

mask_files = glob.glob(f'{PSEUDO_MASK_OUTPUT}/masks/*.png')
print(f'Generated {len(mask_files)} pseudo masks')

if mask_files:
    sample_path = mask_files[0]
    stem = os.path.splitext(os.path.basename(sample_path))[0]
    overlay_path = f'{PSEUDO_MASK_OUTPUT}/overlays/{stem}_fused_layercam.png'

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(np.array(Image.open(sample_path)), cmap='gray')
    axes[0].set_title('Pseudo Mask')
    axes[0].axis('off')

    if os.path.exists(overlay_path):
        axes[1].imshow(np.array(Image.open(overlay_path)))
        axes[1].set_title('Fused LayerCAM Overlay')
        axes[1].axis('off')

    plt.suptitle(f'Sample: {stem}', fontsize=11)
    plt.tight_layout()
    plt.show()

## 4. Stage 3 — U-Net Segmentation Training

Train U-Net (encoder 64→1024) with `0.5 × BCE + 0.5 × Dice` loss on the pseudo masks.

In [ ]:
SEGMENTATION_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/segmentation'

!python train_segmentation.py \
  --data-root "{DATASET_PATH}" \
  --mask-root "{PSEUDO_MASK_OUTPUT}/masks" \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 2 \
  --epochs 25 \
  --output-dir "{SEGMENTATION_OUTPUT}"

In [ ]:
# Plot training curves
import pandas as pd
import matplotlib.pyplot as plt
import os

log_path = f'{SEGMENTATION_OUTPUT}/training_log.csv'
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for ax, metric in zip(axes, ['loss', 'dice', 'iou']):
        ax.plot(df['epoch'], df[f'train_{metric}'], label='train')
        ax.plot(df['epoch'], df[f'val_{metric}'], label='val')
        ax.set_title(metric.capitalize())
        ax.set_xlabel('Epoch')
        ax.legend()

    plt.suptitle('U-Net Training Curves', fontsize=13)
    plt.tight_layout()
    plt.show()

    best = df.loc[df['val_dice'].idxmax()]
    print(f"Best epoch: {int(best['epoch'])} | "
          f"Val Dice: {best['val_dice']:.4f} | "
          f"Val IoU:  {best['val_iou']:.4f}")

## 5. Inference & Visualization

In [ ]:
# Pick a random test image from the dataset
import glob, random, os

all_images = (
    glob.glob(f'{DATASET_PATH}/**/*.jpg', recursive=True) +
    glob.glob(f'{DATASET_PATH}/**/*.jpeg', recursive=True) +
    glob.glob(f'{DATASET_PATH}/**/*.png', recursive=True)
)
TEST_IMAGE = random.choice(all_images)
print(f'Test image: {TEST_IMAGE}')

INFERENCE_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/inference'

!python inference.py \
  --image-path "{TEST_IMAGE}" \
  --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
  --segmentation-checkpoint "{SEGMENTATION_OUTPUT}/best_unet.pt" \
  --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
  --image-size 384 \
  --confidence-threshold 0.5 \
  --cam-percentile 85.0 \
  --max-points 5 \
  --min-component-area 100 \
  --mask-score-threshold 0.4 \
  --closing-kernel 5 \
  --opening-kernel 3 \
  --min-size 200 \
  --output-dir "{INFERENCE_OUTPUT}"

In [ ]:
# Visualize full pipeline output
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

stem = os.path.splitext(os.path.basename(TEST_IMAGE))[0]
panels = {
    'Original':           TEST_IMAGE,
    'Fused LayerCAM':     f'{INFERENCE_OUTPUT}/{stem}_fused_layercam.png',
    'Pseudo Mask (SAM)':  f'{INFERENCE_OUTPUT}/{stem}_pseudo_mask.png',
    'Segmentation Mask':  f'{INFERENCE_OUTPUT}/{stem}_segmentation_mask.png',
    'Final Overlay':      f'{INFERENCE_OUTPUT}/{stem}_final_overlay.png',
}

missing = [title for title, path in panels.items() if not os.path.exists(path)]
if missing:
    print(f'WARNING: missing output files: {missing}')

fig, axes = plt.subplots(1, len(panels), figsize=(20, 4))
for ax, (title, path) in zip(axes, panels.items()):
    if os.path.exists(path):
        ax.imshow(np.array(Image.open(path).convert('RGB')))
    else:
        ax.text(0.5, 0.5, 'Not found', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.suptitle(f'Pipeline Output — {stem}', fontsize=11)
plt.tight_layout()
plt.show()

---
## 6. Debug & Experiments

> Chạy sau khi có `best_classifier.pt`. Không cần train U-Net xong.
>
> Mục tiêu: xác định bottleneck — CAM / Prompt / SAM / Selection — trước khi train full pipeline.

### Experiment Map
| # | Câu hỏi | File xem |
|---|---------|----------|
| E1 | SAM có sinh mask bao phủ toàn bộ xương không? | `debug/<stem>/mask_*.png`, `scores.json` |
| E2 | LayerCAM có cover đủ xương không? | `debug/<stem>/layercam_with_points.png` |
| E3 | Prompt có rơi đúng trên xương không? | `debug/<stem>/foreground.png`, `component_*.png` |
| E4 | Selection method nào tốt nhất? | so sánh `mean` / `sum` / `mean_area` |
| E5 | Batch 20 ảnh đại diện | pipeline strip cho 5 ảnh/class |

In [ ]:
# ── E1 + E2 + E3: Run visualize_pipeline.py with --debug on one image ────────
# Prerequisite: CLASSIFIER_CHECKPOINT and SAM_CHECKPOINT_PATH must be defined above.

import glob, random, os

all_images = (
    glob.glob(f'{DATASET_PATH}/**/*.jpg', recursive=True) +
    glob.glob(f'{DATASET_PATH}/**/*.jpeg', recursive=True) +
    glob.glob(f'{DATASET_PATH}/**/*.png', recursive=True)
)
DEBUG_IMAGE = random.choice(all_images)
print(f'Debug image: {DEBUG_IMAGE}')

DEBUG_VIZ_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/debug_viz'

!python visualize_pipeline.py \
  --image-path "{DEBUG_IMAGE}" \
  --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
  --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
  --image-size 384 \
  --confidence-threshold 0.5 \
  --cam-percentile 85.0 \
  --max-points 5 \
  --min-component-area 100 \
  --mask-score-threshold 0.4 \
  --selection-method mean \
  --debug \
  --output-path "{DEBUG_VIZ_OUTPUT}/pipeline_strip.png"

In [ ]:
# ── E1 + E2 + E3: Display pipeline strip + all debug outputs ─────────────────
import os, json, glob
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

stem = os.path.splitext(os.path.basename(DEBUG_IMAGE))[0]
# debug dir is <output-path parent>/debug/<stem>/ — output-path was DEBUG_VIZ_OUTPUT/pipeline_strip.png
debug_dir_viz = f'{DEBUG_VIZ_OUTPUT}/debug/{stem}'
viz_strip = f'{DEBUG_VIZ_OUTPUT}/pipeline_strip.png'

# --- Panel 1: Pipeline strip ---
if os.path.exists(viz_strip):
    fig, ax = plt.subplots(figsize=(22, 3.5))
    ax.imshow(np.array(Image.open(viz_strip)))
    ax.axis('off')
    ax.set_title('Pipeline Strip', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print(f'WARNING: pipeline strip not found at {viz_strip}')

# --- Panel 2: All SAM candidate masks ---
mask_files = sorted(glob.glob(f'{debug_dir_viz}/mask_*.png'))
overlay_files = sorted(glob.glob(f'{debug_dir_viz}/overlay_mask_*.png'))

if mask_files:
    n = len(mask_files)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    if n == 1:
        axes = [[axes[0]], [axes[1]]]
    for i, (mf, of) in enumerate(zip(mask_files, overlay_files)):
        axes[0][i].imshow(np.array(Image.open(mf)), cmap='gray')
        axes[0][i].set_title(f'mask_{i}', fontsize=9)
        axes[0][i].axis('off')
        if os.path.exists(of):
            axes[1][i].imshow(np.array(Image.open(of).convert('RGB')))
        axes[1][i].set_title(f'overlay_{i}', fontsize=9)
        axes[1][i].axis('off')
    plt.suptitle(f'E1 — SAM Candidate Masks: {stem}', fontsize=11)
    plt.tight_layout()
    plt.show()

    scores_path = f'{debug_dir_viz}/scores.json'
    if os.path.exists(scores_path):
        with open(scores_path) as f:
            scores = json.load(f)
        print('scores.json:')
        for k, v in scores.items():
            bar = '█' * int(v['score'] * 30)
            print(f"  {k}: score={v['score']:.3f}  area={v['area']:6d}  {bar}")
else:
    print(f'No debug masks found. Expected: {debug_dir_viz}')
    print('Make sure --debug flag was passed to visualize_pipeline.py above.')

# --- Panel 3: Prompt + foreground debug ---
fg_path  = f'{debug_dir_viz}/foreground.png'
pts_path = f'{debug_dir_viz}/layercam_with_points.png'
comp_files = sorted(glob.glob(f'{debug_dir_viz}/component_*.png'))

debug_panels = []
if os.path.exists(fg_path):  debug_panels.append(('Foreground (E3)', fg_path))
if os.path.exists(pts_path): debug_panels.append(('Prompts on CAM (E2)', pts_path))
for cf in comp_files[:4]:
    debug_panels.append((os.path.basename(cf), cf))

if debug_panels:
    fig, axes = plt.subplots(1, len(debug_panels), figsize=(5 * len(debug_panels), 4))
    if len(debug_panels) == 1: axes = [axes]
    for ax, (title, path) in zip(axes, debug_panels):
        ax.imshow(np.array(Image.open(path).convert('RGB')))
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    plt.suptitle(f'E2 + E3 — CAM Coverage & Prompts: {stem}', fontsize=11)
    plt.tight_layout()
    plt.show()

### E4 — So sánh Selection Method (`mean` / `sum` / `mean_area`)

Chạy `generate_pseudo_masks.py` 3 lần với selection method khác nhau trên cùng tập ảnh nhỏ, rồi so sánh pseudo mask bằng mắt và Dice/IoU (nếu có GT).

In [ ]:
# ── E4: Compare selection methods on same image ──────────────────────────────
import os, glob
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

E4_BASE = '/content/drive/MyDrive/ThesisOutputs/e4_selection'
METHODS = ['mean', 'sum', 'mean_area']

for method in METHODS:
    out = f'{E4_BASE}/{method}'
    !python visualize_pipeline.py \
      --image-path "{DEBUG_IMAGE}" \
      --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
      --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
      --image-size 384 \
      --confidence-threshold 0.5 \
      --cam-percentile 85.0 \
      --max-points 5 \
      --min-component-area 100 \
      --mask-score-threshold 0.4 \
      --selection-method "{method}" \
      --output-path "{out}/pipeline_strip.png"
    print(f'Done: {method}')

# Display side-by-side comparison
stem = os.path.splitext(os.path.basename(DEBUG_IMAGE))[0]

# Load pseudo mask panel (index 5 in the strip = rightmost 6th panel)
# We compare the full strips
fig, axes = plt.subplots(len(METHODS), 1, figsize=(22, 3.5 * len(METHODS)))
for ax, method in zip(axes, METHODS):
    strip_path = f'{E4_BASE}/{method}/pipeline_strip.png'
    if os.path.exists(strip_path):
        ax.imshow(np.array(Image.open(strip_path)))
        ax.set_ylabel(method, fontsize=12, rotation=0, labelpad=60, va='center')
    else:
        ax.text(0.5, 0.5, f'{method}: not found', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')

plt.suptitle(f'E4 — Selection Method Comparison: {stem}', fontsize=12)
plt.tight_layout()
plt.show()

### E5 — Batch 20 ảnh đại diện (5 per class)

Chạy `visualize_pipeline.py` trên 5 ảnh mỗi class (hand/leg/hip/shoulder). Xuất bảng đánh giá và grid ảnh để xác định bottleneck.

In [ ]:
# ── E5: Batch 20 representative images (5 per anatomy class) ─────────────────
import os, glob, csv, random
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Pick 5 images per class from the dataset CSV
DATASET_CSV = os.path.join(DATASET_PATH, 'dataset.csv')
IMAGE_DIR   = os.path.join(DATASET_PATH, 'images')
CLASSES     = ['hand', 'leg', 'hip', 'shoulder']
N_PER_CLASS = 5
E5_OUTPUT   = '/content/drive/MyDrive/ThesisOutputs/e5_batch20'

# Collect per-class image lists from CSV
per_class: dict[str, list[str]] = {c: [] for c in CLASSES}
with open(DATASET_CSV, encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames or []
    img_col = 'image_id' if 'image_id' in fieldnames else fieldnames[0]
    for row in reader:
        for cls in CLASSES:
            if cls in fieldnames and row.get(cls, '0').strip() in ('1', 'True', 'true'):
                img_path = os.path.join(IMAGE_DIR, row[img_col])
                if not os.path.exists(img_path):
                    # try recursive search
                    found = glob.glob(f'{IMAGE_DIR}/**/{row[img_col]}', recursive=True)
                    if found: img_path = found[0]
                if os.path.exists(img_path):
                    per_class[cls].append(img_path)

random.seed(42)
selected: list[tuple[str, str]] = []  # (class, path)
for cls in CLASSES:
    sample = random.sample(per_class[cls], min(N_PER_CLASS, len(per_class[cls])))
    for p in sample:
        selected.append((cls, p))
    print(f'  {cls}: {len(sample)} images selected (pool={len(per_class[cls])})')

print(f'\nRunning pipeline on {len(selected)} images...')
for cls, img_path in selected:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    out = f'{E5_OUTPUT}/{cls}'
    !python visualize_pipeline.py \
      --image-path "{img_path}" \
      --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
      --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
      --image-size 384 \
      --confidence-threshold 0.5 \
      --cam-percentile 85.0 \
      --max-points 5 \
      --min-component-area 100 \
      --mask-score-threshold 0.4 \
      --selection-method mean \
      --output-path "{out}/{stem}_pipeline.png"

print('Done. Displaying results...')

# Display grid: one row per image, grouped by class
fig_rows = []
for cls in CLASSES:
    strips = sorted(glob.glob(f'{E5_OUTPUT}/{cls}/*_pipeline.png'))
    for p in strips[:N_PER_CLASS]:
        fig_rows.append((cls, p))

if fig_rows:
    fig, axes = plt.subplots(len(fig_rows), 1, figsize=(22, 3.5 * len(fig_rows)))
    if len(fig_rows) == 1: axes = [axes]
    prev_cls = None
    for ax, (cls, path) in zip(axes, fig_rows):
        img = np.array(Image.open(path))
        ax.imshow(img)
        stem = os.path.splitext(os.path.basename(path))[0].replace('_pipeline', '')
        label = f'[{cls}] {stem}' if cls != prev_cls else f'       {stem}'
        ax.set_ylabel(label, fontsize=9, rotation=0, labelpad=120, va='center')
        ax.axis('off')
        prev_cls = cls
    plt.suptitle('E5 — Batch 20 Representative Images', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{E5_OUTPUT}/batch20_grid.png', dpi=80, bbox_inches='tight')
    plt.show()
    print(f'Grid saved to {E5_OUTPUT}/batch20_grid.png')

---
## 7. Bottleneck Analysis (100 ảnh)

> **Mục tiêu:** Trả lời câu hỏi: pipeline đang fail vì SAM hay vì Selection?
>
> - **Task A** — SAM candidate success rate: có ít nhất 1 mask bao phủ đúng xương không?
> - **Task B** — So sánh `mean` / `sum` / `mean_area` trên 100 ảnh, đếm GOOD/PARTIAL/BAD
> - **Task C** — Winner & runner-up từ `scores.json`, phân tích score gap
>
> Chạy **Task A+B+C cùng lúc** — 1 lần pass qua 100 ảnh, lưu kết quả, rồi hiển thị báo cáo.

In [ ]:
# ── Task A + B + C: Run 100-image bottleneck analysis ────────────────────────
# Chạy một lần — lưu kết quả vào ANALYSIS_OUTPUT/results/
# Mỗi ảnh: (a) sinh debug output bằng mean/sum/mean_area, (b) đọc scores.json
#
# Prerequisite: CLASSIFIER_CHECKPOINT, SAM_CHECKPOINT_PATH, DATASET_PATH đã định nghĩa.

import os, glob, csv, json, random, subprocess, sys
from pathlib import Path

ANALYSIS_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/bottleneck_100'
ANALYSIS_N      = 100   # số ảnh khảo sát
METHODS         = ['mean', 'sum', 'mean_area']
CAM_COVER_THRESHOLD = 0.15   # min CAM mean inside mask → mask "covers" xương
RANDOM_SEED     = 42

os.makedirs(ANALYSIS_OUTPUT, exist_ok=True)

# ── 1. Lấy 100 ảnh ngẫu nhiên ────────────────────────────────────────────────
DATASET_CSV = os.path.join(DATASET_PATH, 'dataset.csv')
IMAGE_DIR   = os.path.join(DATASET_PATH, 'images')

all_images = (
    glob.glob(f'{IMAGE_DIR}/**/*.jpg', recursive=True) +
    glob.glob(f'{IMAGE_DIR}/**/*.jpeg', recursive=True) +
    glob.glob(f'{IMAGE_DIR}/**/*.png', recursive=True)
)
random.seed(RANDOM_SEED)
sampled = random.sample(all_images, min(ANALYSIS_N, len(all_images)))
print(f'Sampled {len(sampled)} images from {len(all_images)} total')

# ── 2. Chạy visualize_pipeline.py --debug cho mỗi ảnh, mỗi method ────────────
# Lưu strips vào ANALYSIS_OUTPUT/<method>/<stem>_pipeline.png
# Lưu debug (scores.json) vào ANALYSIS_OUTPUT/<method>/debug/<stem>/

for i, img_path in enumerate(sampled):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    print(f'[{i+1:3d}/{len(sampled)}] {stem}', end='  ')
    for method in METHODS:
        out_dir = f'{ANALYSIS_OUTPUT}/{method}'
        strip_path = f'{out_dir}/{stem}_pipeline.png'
        if os.path.exists(strip_path):
            print(f'{method}:cached', end='  ')
            continue
        ret = subprocess.run(
            [sys.executable, 'visualize_pipeline.py',
             '--image-path', img_path,
             '--classifier-checkpoint', CLASSIFIER_CHECKPOINT,
             '--sam-checkpoint', SAM_CHECKPOINT_PATH,
             '--image-size', '384',
             '--confidence-threshold', '0.5',
             '--cam-percentile', '85.0',
             '--max-points', '5',
             '--min-component-area', '100',
             '--mask-score-threshold', '0.4',
             '--selection-method', method,
             '--debug',
             '--output-path', strip_path],
            capture_output=True, text=True
        )
        ok = '✓' if ret.returncode == 0 else '✗'
        print(f'{method}:{ok}', end='  ')
        if ret.returncode != 0:
            print(f'\n  ERROR: {ret.stderr[-300:]}', end='')
    print()

print('\nAll runs complete.')
print(f'Results in: {ANALYSIS_OUTPUT}/')

In [ ]:
# ── Task A + B + C: Analyse results ──────────────────────────────────────────
# Đọc debug outputs, tính:
#   Task A — SAM candidate success rate (YES nếu SAM iou score >= ngưỡng)
#   Task B — so sánh pseudo mask AREA theo method (proxy coverage; scores.json
#             chứa SAM's own iou scores — giống nhau với mọi selection method,
#             không dùng được. Dùng pixel area của last panel trong strip.)
#   Task C — winner / runner-up per image (SAM iou scores), score gap

import os, json, glob
import numpy as np
from PIL import Image
from collections import defaultdict

METHODS = ['mean', 'sum', 'mean_area']
SAM_SCORE_THRESHOLD = 0.7    # task A: SAM iou score >= này → candidate "good"
GOOD_AREA_FRAC      = 0.05   # task B: mask area >= 5% image → GOOD
PARTIAL_AREA_FRAC   = 0.01   # task B: >= 1% → PARTIAL, else BAD
IMAGE_PIXELS        = 384 * 384

# Collect all stems from the 'mean' run (reference)
ref_strips = sorted(glob.glob(f'{ANALYSIS_OUTPUT}/mean/*_pipeline.png'))
stems = [os.path.basename(p).replace('_pipeline.png', '') for p in ref_strips]
print(f'Analysing {len(stems)} images...\n')

task_a_results = {}
task_b_results = defaultdict(dict)
task_c_results = {}

for stem in stems:
    # ── Task A: SAM iou scores from 'mean' debug dir ─────────────────────────
    scores_path = f'{ANALYSIS_OUTPUT}/mean/debug/{stem}/scores.json'
    if not os.path.exists(scores_path):
        task_a_results[stem] = 'NO (no debug)'
        for m in METHODS:
            task_b_results[stem][m] = 'N/A'
        continue

    with open(scores_path) as f:
        scores_data = json.load(f)

    sam_scores = {k: v['score'] for k, v in scores_data.items()}
    sorted_scores = sorted(sam_scores.items(), key=lambda x: x[1], reverse=True)

    max_score = sorted_scores[0][1] if sorted_scores else 0.0
    task_a_results[stem] = 'YES' if max_score >= SAM_SCORE_THRESHOLD else 'NO'

    # Task C: winner / runner-up
    winner = sorted_scores[0] if sorted_scores else ('none', 0.0)
    runner = sorted_scores[1] if len(sorted_scores) >= 2 else ('none', 0.0)
    task_c_results[stem] = {
        'winner': winner[0], 'winner_score': winner[1],
        'runner_up': runner[0], 'runner_score': runner[1],
        'gap': winner[1] - runner[1], 'n_masks': len(sorted_scores),
    }

    # ── Task B: measure pseudo mask area from last panel of pipeline strip ───
    # scores.json stores SAM iou scores (same for all methods), not CAM scores.
    # Instead, read the rightmost 1/6 of the pipeline strip as proxy for coverage.
    for method in METHODS:
        strip_path = f'{ANALYSIS_OUTPUT}/{method}/{stem}_pipeline.png'
        if not os.path.exists(strip_path):
            task_b_results[stem][method] = 'N/A'
            continue
        strip = np.array(Image.open(strip_path).convert('L'))
        panel_w = strip.shape[1] // 6
        last_panel = strip[:, -panel_w:]
        bone_pixels = int((last_panel[22:] > 127).sum())  # skip 22px label header
        area_frac = bone_pixels / IMAGE_PIXELS
        if area_frac >= GOOD_AREA_FRAC:
            task_b_results[stem][method] = 'GOOD'
        elif area_frac >= PARTIAL_AREA_FRAC:
            task_b_results[stem][method] = 'PARTIAL'
        else:
            task_b_results[stem][method] = 'BAD'

# ── Print Task A ──────────────────────────────────────────────────────────────
yes_count = sum(1 for v in task_a_results.values() if v == 'YES')
no_count  = sum(1 for v in task_a_results.values() if v.startswith('NO'))
total     = len(task_a_results)
print('═' * 60)
print('TASK A — SAM Candidate Success Rate')
print('═' * 60)
print(f'  YES (SAM iou score >= {SAM_SCORE_THRESHOLD}): {yes_count}/{total}  ({yes_count/total*100:.1f}%)')
print(f'  NO:                                            {no_count}/{total}  ({no_count/total*100:.1f}%)')
print()

# ── Print Task B ──────────────────────────────────────────────────────────────
print('═' * 60)
print('TASK B — Pseudo Mask Coverage by Selection Method')
print(f'  GOOD >= {GOOD_AREA_FRAC*100:.0f}% px | PARTIAL >= {PARTIAL_AREA_FRAC*100:.0f}% px | BAD otherwise')
print('═' * 60)
header = f"{'stem':<30}" + ''.join(f'{m:>12}' for m in METHODS)
print(header)
print('-' * (30 + 12 * len(METHODS)))

method_counts = {m: defaultdict(int) for m in METHODS}
for stem in stems:
    row = f'{stem[:29]:<30}'
    for m in METHODS:
        rating = task_b_results[stem].get(m, 'N/A')
        row += f'{rating:>12}'
        method_counts[m][rating] += 1
    print(row)

print('-' * (30 + 12 * len(METHODS)))
summary_row = f"{'TOTAL':<30}"
for m in METHODS:
    c = method_counts[m]
    summary_row += f"G:{c['GOOD']} P:{c['PARTIAL']} B:{c['BAD']}".rjust(12)
print(summary_row)
print()

# ── Print Task C ──────────────────────────────────────────────────────────────
print('═' * 60)
print('TASK C — SAM Candidate Winner & Runner-up (by SAM iou score)')
print('═' * 60)
sorted_by_gap = sorted(task_c_results.items(), key=lambda x: x[1]['gap'], reverse=True)
print(f"{'stem':<30}  {'winner':<8}  {'score':>6}  {'runner':>8}  {'score':>6}  {'gap':>6}  {'n':>3}")
print('-' * 75)
for stem, r in sorted_by_gap[:50]:
    print(f"{stem[:29]:<30}  {r['winner']:<8}  {r['winner_score']:>6.3f}"
          f"  {r['runner_up']:>8}  {r['runner_score']:>6.3f}  {r['gap']:>6.3f}  {r['n_masks']:>3}")

results_path = f'{ANALYSIS_OUTPUT}/bottleneck_results.json'
with open(results_path, 'w') as f:
    json.dump({
        'task_a': task_a_results,
        'task_b': {s: dict(v) for s, v in task_b_results.items()},
        'task_c': task_c_results,
        'summary': {
            'sam_success_rate': yes_count / total if total else 0,
            'method_counts': {m: dict(method_counts[m]) for m in METHODS},
        }
    }, f, indent=2)
print(f'\nFull results saved to {results_path}')

---
## 8. Bước 2 — Multi-Mask Fusion

> Thay vì chọn 1 mask tốt nhất, lấy **top-k masks** và hợp lại (union hoặc intersection).
>
> - **top3_union**: `mask1 | mask2 | mask3` — bao phủ nhiều hơn, phù hợp khi CAM cover thấp
> - **top2_intersection**: `mask1 & mask2` — chính xác hơn, phù hợp khi SAM score gap lớn
>
> Cần thêm `--fusion-topk` vào `select_and_fuse_masks` trong `pseudo/mask_selection.py`.

In [ ]:
# ── Bước 2: Multi-Mask Fusion — so sánh các chiến lược kết hợp mask ──────────
# fusion_topk:
#   0  → OR của tất cả mask >= threshold (default cũ)
#   3  → union top-3 masks (covering)
#   -2 → intersection top-2 masks (precise)

import subprocess, sys, os, glob
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

FUSION_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/fusion_compare'
FUSION_IMAGE  = DEBUG_IMAGE   # same image as E1-E4

FUSION_STRATEGIES = [
    ('default_OR',          ['--selection-method', 'mean',       '--fusion-topk', '0']),
    ('top3_union_mean',     ['--selection-method', 'mean',       '--fusion-topk', '3']),
    ('top2_intersect_mean', ['--selection-method', 'mean',       '--fusion-topk', '-2']),
    ('top3_union_meanarea', ['--selection-method', 'mean_area',  '--fusion-topk', '3']),
]

os.makedirs(FUSION_OUTPUT, exist_ok=True)

for label, extra_args in FUSION_STRATEGIES:
    out_path = f'{FUSION_OUTPUT}/{label}_pipeline.png'
    if os.path.exists(out_path):
        print(f'{label}: cached')
        continue
    ret = subprocess.run(
        [sys.executable, 'visualize_pipeline.py',
         '--image-path', FUSION_IMAGE,
         '--classifier-checkpoint', CLASSIFIER_CHECKPOINT,
         '--sam-checkpoint', SAM_CHECKPOINT_PATH,
         '--image-size', '384',
         '--confidence-threshold', '0.5',
         '--cam-percentile', '85.0',
         '--max-points', '5',
         '--min-component-area', '100',
         '--mask-score-threshold', '0.4',
         '--output-path', out_path] + extra_args,
        capture_output=True, text=True
    )
    if ret.returncode == 0:
        print(f'{label}: ✓')
    else:
        print(f'{label}: ✗')
        print(ret.stderr[-300:])

# Display
fig, axes = plt.subplots(len(FUSION_STRATEGIES), 1, figsize=(22, 3.5 * len(FUSION_STRATEGIES)))
if len(FUSION_STRATEGIES) == 1:
    axes = [axes]
for ax, (label, _) in zip(axes, FUSION_STRATEGIES):
    p = f'{FUSION_OUTPUT}/{label}_pipeline.png'
    if os.path.exists(p):
        ax.imshow(np.array(Image.open(p)))
        ax.set_ylabel(label, fontsize=10, rotation=0, labelpad=130, va='center')
    else:
        ax.text(0.5, 0.5, f'{label}: not found', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')
plt.suptitle('Bước 2 — Fusion Strategy Comparison', fontsize=12)
plt.tight_layout()
plt.show()

---
## 9. Bước 3 — Train U-Net với Best Strategy

> Sau khi có kết quả Task A/B, chọn `BEST_METHOD` và `BEST_FUSION_TOPK` tốt nhất,
> regen pseudo masks cho toàn bộ dataset, train U-Net, so sánh Dice/IoU.

In [ ]:
# ── Bước 3: Regenerate pseudo masks với best strategy, train U-Net ────────────
# Sau khi phân tích Task A/B, điều chỉnh 2 dòng này:
BEST_METHOD      = 'mean'      # 'mean' | 'sum' | 'mean_area'
BEST_FUSION_TOPK = 0           # 0=OR-all, 3=union-top3, -2=intersect-top2

BEST_MASKS_OUTPUT = f'/content/drive/MyDrive/ThesisOutputs/pseudo_masks_best_{BEST_METHOD}_topk{BEST_FUSION_TOPK}'
BEST_SEG_OUTPUT   = f'/content/drive/MyDrive/ThesisOutputs/segmentation_best_{BEST_METHOD}_topk{BEST_FUSION_TOPK}'

print(f'Strategy: selection={BEST_METHOD}, fusion_topk={BEST_FUSION_TOPK}')
print(f'Pseudo masks → {BEST_MASKS_OUTPUT}')
print(f'U-Net output → {BEST_SEG_OUTPUT}')

# ── Step 3a: Generate pseudo masks ───────────────────────────────────────────
!python generate_pseudo_masks.py \
  --data-root "{DATASET_PATH}" \
  --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
  --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
  --target-columns hand,leg,hip,shoulder \
  --image-size 384 \
  --batch-size 1 \
  --num-workers 2 \
  --confidence-threshold 0.5 \
  --cam-percentile 85.0 \
  --max-points 5 \
  --min-component-area 100 \
  --mask-score-threshold 0.4 \
  --selection-method "{BEST_METHOD}" \
  --closing-kernel 5 \
  --opening-kernel 3 \
  --min-size 200 \
  --output-dir "{BEST_MASKS_OUTPUT}"

In [ ]:
# ── Step 3b: Train U-Net với best pseudo masks ────────────────────────────────
!python train_segmentation.py \
  --data-root "{DATASET_PATH}" \
  --mask-root "{BEST_MASKS_OUTPUT}/masks" \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 2 \
  --epochs 25 \
  --output-dir "{BEST_SEG_OUTPUT}"

# ── Step 3c: Compare training curves: original vs best strategy ───────────────
import pandas as pd
import matplotlib.pyplot as plt
import os

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

runs = [
    ('Original (mean, OR-all)', SEGMENTATION_OUTPUT),
    (f'Best ({BEST_METHOD}, topk={BEST_FUSION_TOPK})', BEST_SEG_OUTPUT),
]

colors = ['steelblue', 'tomato']
for metric, ax in zip(['loss', 'dice', 'iou'], axes):
    for (label, out_dir), color in zip(runs, colors):
        log = f'{out_dir}/training_log.csv'
        if os.path.exists(log):
            df = pd.read_csv(log)
            ax.plot(df['epoch'], df[f'val_{metric}'], label=label, color=color)
    ax.set_title(f'Val {metric.capitalize()}')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=7)

plt.suptitle('Bước 3 — U-Net: Original vs Best Strategy', fontsize=12)
plt.tight_layout()
plt.show()

# ── Print summary table ────────────────────────────────────────────────────────
print(f'\n{"Run":<40}  {"Val Dice":>9}  {"Val IoU":>9}')
print('-' * 62)
for label, out_dir in runs:
    log = f'{out_dir}/training_log.csv'
    if os.path.exists(log):
        df = pd.read_csv(log)
        best = df.loc[df['val_dice'].idxmax()]
        print(f'{label:<40}  {best["val_dice"]:>9.4f}  {best["val_iou"]:>9.4f}')
    else:
        print(f'{label:<40}  {"N/A":>9}  {"N/A":>9}')